# CodeAlpha Data Science Internship: Sales Prediction using Python
**Author**: Data Science Intern
**Repository**: CodeAlpha_SalesPrediction
**Dataset**: ISLR Advertising Benchmark Dataset (Marketing Mix Modeling)

## Executive Objective
Analyze historical advertising expenditures across TV, Radio, and Newspaper channels, train and cross-validate predictive machine learning models to forecast product sales, inspect residual diagnostics, and formulate evidence-based business insights while respecting causal inference boundaries.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Data Ingestion & Exploratory Data Analysis (EDA)
We load `data/advertising.csv` and inspect data shapes, data types, missing values, and descriptive statistics.

In [2]:
df = pd.read_csv('../data/advertising.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
print('Dataset shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
df.describe()

### Correlation Matrix & Pairwise Inspection
Let us examine how advertising spend correlates with sales and whether collinearity exists between channels.

In [3]:
plt.figure(figsize=(6, 4))
sns.heatmap(df.corr(), annot=True, cmap='Blues', fmt='.3f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap: Advertising Spend vs Sales')
plt.tight_layout()
plt.show()

## 2. Model Design & Preprocessing Pipeline
Because the 200 market records represent independent cross-sectional regional observations (not time-indexed sequential transactions), cross-validation with shuffled folds is statistically appropriate.

We divide the dataset into an 80% training set and a 20% holdout test set with a fixed random seed (`random_state=42`).

In [4]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')

## 3. Candidate Model Benchmarking (5-Fold Cross Validation)
We compare:
1. **Dummy Baseline Regressor** (predicts training target mean)
2. **OLS Linear Regression**
3. **Ridge Regression** (L2 regularized)
4. **Random Forest Regressor**
5. **HistGradientBoosting Regressor**

In [5]:
models = {
    'Baseline (Mean)': DummyRegressor(strategy='mean'),
    'OLS Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=100, max_depth=4, random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    cv_res = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring={'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results.append({
        'Model': name,
        'CV MAE': -cv_res['test_mae'].mean(),
        'CV RMSE': -cv_res['test_rmse'].mean(),
        'CV R2': cv_res['test_r2'].mean(),
        'Holdout RMSE': root_mean_squared_error(y_test, y_pred),
        'Holdout R2': r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df.round(4)

## 4. Residual Diagnostics & Evaluation Plots
We inspect predicted vs. actual values and residual distributions for the best performing model.

In [6]:
best_model = models['HistGradientBoosting']
y_test_pred = best_model.predict(X_test)
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Predicted vs Actual
axes[0].scatter(y_test, y_test_pred, color='#0284c7', alpha=0.8, edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=1.5)
axes[0].set_xlabel('Actual Sales (1,000s)')
axes[0].set_ylabel('Predicted Sales (1,000s)')
axes[0].set_title('Predicted vs. Actual (Holdout Set)')

# Residual distribution
sns.histplot(residuals, kde=True, ax=axes[1], color='#38bdf8', bins=15)
axes[1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Error Distribution')

plt.tight_layout()
plt.show()

## 5. Evidence-Based Business Findings
1. **TV Spend is the Anchor**: High univariate and multivariate significance.
2. **Radio Provides Strongest Per-Dollar Marginal Lift**: But experiences saturation above ~$40k.
3. **Newspaper Advertising is Redundant**: Controlled multivariate p-value is > 0.8; budget can likely be reallocated.
4. **Causal Caution**: Observational correlation does not prove causation; run geo-matched randomized holdout experiments prior to large-scale capital redeployment.